<a href="https://colab.research.google.com/github/rvnmina/ASD-Facial-Image-Detection-CNN/blob/main/Adaptive_Rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%%writefile paths.py
import os
HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.environ.get(
    "ARB_ROOT",
    os.path.dirname(HERE) if os.path.basename(HERE) == "code" else HERE
)
RUNS  = os.environ.get("ARB_OUT",   os.path.join(ROOT, "runs"))
RES   = os.environ.get("ARB_RES",   os.path.join(ROOT, "results"))
FIG   = os.environ.get("ARB_FIG",   os.path.join(ROOT, "figures"))
CACHE = os.environ.get("ARB_CACHE", os.path.join(ROOT, "cache"))
PV_ROOT = os.environ.get(
    "PV_ROOT",
    os.path.join(ROOT, "data", "PlantVillage-Dataset", "raw", "color")
)

Overwriting paths.py


In [7]:
import paths
print("ROOT =", paths.ROOT)
print("RUNS =", paths.RUNS)

ROOT = /content
RUNS = /content/runs


In [12]:
%%writefile p3_core.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RandomFeatureMap(nn.Module):

    def __init__(self, d, r):
        super().__init__()
        self.register_buffer("W", torch.randn(r, d))
        self.register_buffer("b", torch.rand(r) * 2 * math.pi)
        m = torch.arange(1, r + 1, dtype=torch.float32)
        self.register_buffer("m_over_r", m / r)
        self.scale = math.sqrt(2.0 / r)

    def forward(self, x):
        proj = x @ self.W.t()
        proj = 2.0 * math.pi * self.m_over_r * proj
        phi = torch.cos(proj + self.b) * self.scale

        return phi
        #paper bounded attetion layer
class P3Attention(nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.d = d
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.phi = RandomFeatureMap(d, r)

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Phi_Q = self.phi(Q)
        Phi_K = self.phi(K)

        #Compute Z
        KtV = torch.bmm(Phi_K.transpose(1, 2), V)
        Z = torch.bmm(Phi_Q, KtV)
        return Z


#Transformer Encoder Block Attention + Feed-forward + Layer Norms
class P3Block(nn.Module):
    def __init__(self, d, r, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = P3Attention(d, r)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, d),
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


#full Vision Transformer with P-3 attention Image -patches - transformer - classification
class P3ViT(nn.Module):
    #Vision Transformer using P-3 bounded-rank attention.
    def __init__(self, img_size=96, patch_size=8, in_chans=3,
                 num_classes=5, d=128, depth=12, mlp_dim=256, r=32):
        super().__init__()
        self.patch_size = patch_size
        num_patches = (img_size // patch_size) ** 2
        patch_dim = in_chans * patch_size ** 2
        self.patch_embed = nn.Linear(patch_dim, d)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d))
        self.blocks = nn.Sequential(
            *[P3Block(d, r, mlp_dim) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x):
        B = x.shape[0]
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5)
        x = x.reshape(B, -1, 3 * p * p)

        #project patches to d dimensions
        x = self.patch_embed(x)
        #prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        #add position embeddings
        x = x + self.pos_embed
        #pass through 12 transformer blocks
        x = self.blocks(x)
        #take CLS token output
        x = self.norm(x[:, 0])
        x = self.head(x)
        return x
#SOFTMAX VIT
class SoftmaxAttention(nn.Module):
    #Standard softmax attention: A = softmax(QK^T/√d) · V
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.scale = d ** -0.5

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        attn = torch.softmax(Q @ K.transpose(-2, -1) * self.scale, dim=-1)
        return attn @ V

class SoftmaxBlock(nn.Module):
    def __init__(self, d, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = SoftmaxAttention(d)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, d),
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x
class SoftmaxViT(nn.Module):
    #Standard ViT with softmax attention — the expensive baseline.
    def __init__(self, img_size=96, patch_size=8, in_chans=3,
                 num_classes=5, d=128, depth=12, mlp_dim=256):
        super().__init__()
        self.patch_size = patch_size
        num_patches = (img_size // patch_size) ** 2
        patch_dim = in_chans * patch_size ** 2
        self.patch_embed = nn.Linear(patch_dim, d)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d))
        self.blocks = nn.Sequential(
            *[SoftmaxBlock(d, mlp_dim) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x):
        B = x.shape[0]
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5).reshape(B, -1, 3 * p * p)
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.blocks(x)
        x = self.norm(x[:, 0])
        return self.head(x)

Overwriting p3_core.py


In [13]:
import importlib, torch, p3_core
importlib.reload(p3_core)

model = p3_core.P3ViT(r=32)
dummy = torch.randn(1, 3, 96, 96)
out = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Input:  torch.Size([1, 3, 96, 96])
Output: torch.Size([1, 5])
Params: 1,431,301
